# 05 — CHW Triage Demo: From Community Data to Actionable Referrals

This notebook demonstrates the end-to-end pipeline from Prompt B (signal from
community data) to Prompt C (closing the diagnosis gap).

**Scenario:** A community health worker collects self-reported menstrual health
data from women in a low-resource setting. The triage tool processes this data
and produces a differential diagnosis — no lab, no imaging, no internet.

**Everything here is synthetic.** Distributions are simulated. Do not interpret
as real prevalence or clinical evidence.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from menstrual_health_open.synthetic import generate_records
from menstrual_health_open.triage import triage_record, triage_summary
from menstrual_health_open.risk_model import score_all_conditions, risk_level

%matplotlib inline

## 1. Generate a community dataset

We simulate data collection from a community of 200 women.

In [ ]:
records = generate_records(200, seed=99, include_conditions=True)
df = pd.DataFrame(records)

print(f"Community size: {len(df)} women")
print(f"Settings: {df['setting'].value_counts().to_dict()}")
print()
print("Ground truth prevalence (synthetic):")
for col in ['condition_iron_deficiency', 'condition_fibroids', 'condition_coagulation_disorder']:
    label = col.replace('condition_', '').replace('_', ' ').title()
    pct = df[col].eq('yes').mean() * 100
    print(f"  {label}: {pct:.1f}%")

## 2. Triage an individual patient

This is what a CHW would see for a single woman.

In [ ]:
idx = df[df['condition_iron_deficiency'] == 'yes'].index[0]
record = df.iloc[idx].to_dict()

print(triage_summary(record))
print()
print(f"Ground truth: Iron Deficiency = {record['condition_iron_deficiency']}, "
      f"Fibroids = {record['condition_fibroids']}, "
      f"Coagulation = {record['condition_coagulation_disorder']}")

## 3. Triage the entire community

Process all 200 records and look at the risk distribution across the community.

In [ ]:
risk_scores = []
for _, row in df.iterrows():
    scores = score_all_conditions(row.to_dict())
    scores['record_id'] = row['record_id']
    scores['setting'] = row['setting']
    risk_scores.append(scores)

risk_df = pd.DataFrame(risk_scores)

print("Risk score distribution across the community:")
for cond in ['iron_deficiency', 'fibroids_adenomyosis', 'coagulation_disorder']:
    col = risk_df[cond]
    high = (col >= 70).sum()
    mod = ((col >= 40) & (col < 70)).sum()
    low = (col < 40).sum()
    print(f"  {cond:25s} high={high:3d} moderate={mod:3d} low={low:3d}")

In [ ]:
risk_long = risk_df.melt(
    id_vars=['record_id', 'setting'],
    value_vars=['iron_deficiency', 'fibroids_adenomyosis', 'coagulation_disorder'],
    var_name='condition', value_name='score'
)

condition_labels = {
    'iron_deficiency': 'Iron Deficiency',
    'fibroids_adenomyosis': 'Fibroids / Adenomyosis',
    'coagulation_disorder': 'Coagulation Disorder',
}
risk_long['label'] = risk_long['condition'].map(condition_labels)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (cond, label) in zip(axes, condition_labels.items()):
    data = risk_long[risk_long['condition'] == cond]
    bp = ax.boxplot([data[data['setting'] == s]['score'] for s in ['urban', 'peri_urban', 'rural']],
                    labels=['Urban', 'Peri-urban', 'Rural'])
    ax.set_title(label)
    ax.set_ylabel('Risk score (0-100)')
    ax.axhline(70, color='red', linestyle='--', alpha=0.4, label='High risk threshold')
    ax.axhline(40, color='orange', linestyle='--', alpha=0.4, label='Moderate risk threshold')
plt.tight_layout()
plt.show()

## 4. Triage accuracy vs ground truth

How well does the rule-based triage tool identify the true condition?

In [ ]:
def correct_triage(row) -> bool:
    true_conditions = []
    if row['condition_iron_deficiency'] == 'yes':
        true_conditions.append('iron_deficiency')
    if row['condition_fibroids'] == 'yes':
        true_conditions.append('fibroids_adenomyosis')
    if row['condition_coagulation_disorder'] == 'yes':
        true_conditions.append('coagulation_disorder')

    if not true_conditions:
        return True

    scores = score_all_conditions(row.to_dict())
    top = max(scores, key=scores.get)
    return top in true_conditions


correct = sum(correct_triage(row) for _, row in df.iterrows())
print(f"Triage selects a true condition as top concern: {correct / len(df):.1%}")

has_condition = df[
    (df['condition_iron_deficiency'] == 'yes') |
    (df['condition_fibroids'] == 'yes') |
    (df['condition_coagulation_disorder'] == 'yes')
]
correct_with_cond = sum(correct_triage(row) for _, row in has_condition.iterrows())
print(f"Among women with at least one condition: {correct_with_cond / len(has_condition):.1%}")

## 5. NLP symptom extraction from free text

Prompt B asks for NLP pipelines. The synthetic data includes `symptom_free_text`
field with simulated free-text descriptions. We demonstrate a lightweight
keyword-based extractor that maps free text to the controlled vocabulary.

This pattern would work in the field: a CHW types or speaks the woman's
description, and the tool extracts structured symptoms for the triage model.

In [ ]:
SYMPTOM_KEYWORDS = {
    'cramps': ['cramp', 'pain', 'ache', 'cramping'],
    'fatigue': ['tired', 'exhausted', 'fatigue', 'energy', 'sleep', 'low energy'],
    'dizziness': ['dizzy', 'lightheaded', 'faint', 'dizziness', 'spells'],
    'headache': ['headache', 'head', 'migraine'],
    'nausea': ['nausea', 'sick', 'vomit'],
    'back_pain': ['back pain', 'back ache', 'back', 'lower back'],
    'mood_changes': ['mood', 'irritable', 'emotional', 'cry'],
    'heavy_bleeding': ['heavy', 'clots', 'soaking', 'flood', 'pad'],
}

def extract_symptoms(free_text: str) -> list[str]:
    """Extract controlled-vocabulary symptoms from free text."""
    text = free_text.lower()
    found = []
    for symptom, keywords in SYMPTOM_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                found.append(symptom)
                break
    return sorted(found)


texts_with_data = df[df['symptom_free_text'].str.len() > 0]
sample = texts_with_data.iloc[0]

print(f"Free text: {sample['symptom_free_text']}")
print(f"Extracted symptoms: {extract_symptoms(sample['symptom_free_text'])}")
print(f"Original reported_symptoms: {sample['reported_symptoms']}")

In [ ]:
total = 0
partial = 0
full_mismatch = 0

for _, row in texts_with_data.iterrows():
    original = set(s.strip() for s in str(row['reported_symptoms']).split(';') if s.strip())
    extracted = set(extract_symptoms(str(row['symptom_free_text'])))

    if not original:
        continue
    total += 1
    overlap = original & extracted
    if len(overlap) >= len(original):
        partial += 1
    elif len(overlap) == 0:
        full_mismatch += 1

print(f"Free-text records with original symptoms: {total}")
print(f"Full match (all symptoms extracted):       {partial} ({partial/total:.0%})")
print(f"Partial/no match:                          {total - partial} ({(total - partial)/total:.0%})")
print(f"Full mismatch:                              {full_mismatch} ({full_mismatch/total:.0%})")
print()
print("The keyword matcher is a starting point. Production systems would use")
print("a proper NLP pipeline with entity recognition, spelling correction,")
print("and multilingual support.")

## Summary

This notebook demonstrates the end-to-end pipeline:

1. **Data collection** — Self-reported menstrual health data from a community
2. **Signal extraction (B)** — Risk scoring models surface patterns from noisy data
3. **Triage (C)** — Differential diagnosis with referral recommendations
4. **NLP** — Free-text symptom -> controlled vocabulary mapping
5. **Offline-capable** — Pure-Python scoring, no internet required

**What this enables:** A CHW with a simple mobile form or paper questionnaire
can collect data, get instant triage guidance, and make informed referral
decisions — all without specialist equipment or clinical infrastructure.